**TRC3500 Project 2 - Coin Drop Detection and Classification**


In [ ]:
# Import modules

import serial                           # For serial communication
import numpy as np                      # For data processing
import matplotlib.pyplot as plt         # For plotting
from scipy.signal import find_peaks     # For signal peaks extractions

In [ ]:
# Define Parameters

# Serial communications params
PORT = 'COM10'
BAUD = 115200
SAMPLES = 5000
BYTES_PER_SAMPLE = 2
EXPECTED_BYTES = SAMPLES * BYTES_PER_SAMPLE

# ADC configurations
VREF = 3.3
ADC_MAX = 4095
FS = 10000  # 10 kHz sampling rate

In [ ]:
# ALGORITHM TUNING PARAMETERS - for peak extraction
PEAK_HEIGHT_THRESHOLD = 0.04  # Minimum voltage (above average) to count as a peak/bounce
PEAK_MIN_DISTANCE = 500  # Min samples between distinct peaks (500 samples = 50ms)

In [ ]:
def classify_impact_with_dist(peak_amplitude, num_peaks, peak_dist):
    """
    Classifier for all 4 cases based on the peak amplitude, number of peaks and peak distance. The cases are:
        Case A - 30cm Height, 10cm Distance
        Case B - 10cm Height, 10cm Distance
        Case C - 30cm Height, 30cm Distance
        Case D - 10cm Height, 30cm Distance
    """
    # Class A logic
    if ((0.751 <= peak_amplitude <= 2.5) and (1 <= num_peaks <= 3) and ((peak_dist > 2000) or (peak_dist == 0))):
        return "Class A -> Height: 30cm and Distance: 10cm"
    
    # Class B logic
    elif ((0.4 <= peak_amplitude <= 1.3) and (2 <= num_peaks <= 6) and (peak_dist <= 2000)):
        return "Class B -> Height: 10cm and Distance: 10cm"
            
    # Class C logic 
    elif ((0.46 <= peak_amplitude <= 0.75) and (1 <= num_peaks <= 3) and ((peak_dist > 2000) or (peak_dist == 0))):
        return "Class C -> Height: 30cm and Distance: 30cm"
            
    # Class D logic
    elif ((0.05 <= peak_amplitude <= 0.4) and (num_peaks >= 1)):
        return "Class D -> Height: 10cm and Distance: 30cm"
        
    else:
        return "IDK"

In [ ]:
def extract_features(voltage_signal):
    """
    Analyses the raw voltage array to calculate peak_amplitude, num_peaks, and peak_distance
    """
    # Remove DC offset
    mean_voltage = np.mean(voltage_signal)
    signal_ac = voltage_signal - mean_voltage
    abs_signal = np.abs(signal_ac)

    # Find Peaks (Bounces)
    peaks, _ = find_peaks(abs_signal, height=PEAK_HEIGHT_THRESHOLD, distance=PEAK_MIN_DISTANCE)
    num_peaks = len(peaks)

    # Calculate Distance between first two peaks
    peak_dist = 0
    if num_peaks >= 2:
        # Distance in samples between the first and second peak
        peak_dist = peaks[1] - peaks[0]

    # Calculate Peak Amplitude
    if num_peaks > 0:
        peak_amplitude = np.max(abs_signal[peaks])
    else:
        peak_amplitude = np.max(abs_signal)

    return peak_amplitude, num_peaks, peaks, abs_signal, peak_dist

In [ ]:
def main():
    # Setup interactive plotting
    plt.ion()
    fig, ax = plt.subplots(figsize=(10, 5))
    t = np.arange(SAMPLES) / FS

    try:
        ser = serial.Serial(PORT, BAUD, timeout=3)
        print(f"Connected to {PORT}. Waiting for raw data block ({EXPECTED_BYTES} bytes)...")

        # Flush any garbage data remaining in the buffer
        ser.reset_input_buffer()

        while True:
            # Wait until we have a full capture in the buffer
            if ser.in_waiting >= EXPECTED_BYTES:

                # Read raw binary data
                data = ser.read(EXPECTED_BYTES)

                if len(data) == EXPECTED_BYTES:
                    # Convert bytes to voltage array
                    samples = np.frombuffer(data, dtype='<u2')
                    voltage = samples * (VREF / ADC_MAX)

                    # Extract the required features for peak extraction
                    peak_amplitude, num_peaks, peaks, abs_sig, peak_dist = extract_features(voltage)

                    # Classify the extracted peaks 
                    result = classify_impact_with_dist(peak_amplitude, num_peaks, peak_dist)

                    # Print results to console
                    print(f"Peaks: {num_peaks} | Max Amplitude: {peak_amplitude:.3f}V | Peak_dist: {peak_dist:.3f} | RESULT: {result}")

                    # Update Plot for visuals
                    ax.clear()
                    ax.plot(t, abs_sig, label="Absolute Signal (AC)")

                    # Mark detected peaks with a red 'x'
                    if num_peaks > 0:
                        ax.plot(t[peaks], abs_sig[peaks], "rx", markersize=10, label="Detected Peaks")

                    ax.set_title(f"Classified: {result}\nPeaks: {num_peaks}, Amplitude: {peak_amplitude:.3f}V")
                    ax.set_xlabel("Time (s)")
                    ax.set_ylabel("Voltage Magnitude (V)")
                    ax.legend()
                    ax.grid()

                    # Pause allows the plot window to refresh
                    plt.pause(0.01)

                    # Flush buffer so we wait for the NEXT fresh event cleanly
                    ser.reset_input_buffer()

    except serial.SerialException as e:
        print(f"Serial Error: {e}")
    except KeyboardInterrupt:
        print("\nStopping script...")
    finally:
        if 'ser' in locals() and ser.is_open:
            ser.close()
        plt.ioff()
        plt.show()

In [ ]:
# Run the main code

main()